# Corrected analysis: Olist delivery and reviews

The outcome is the order's review score, and the exposure is delivery after the estimated date. The point estimates are descriptive of delivered orders with a review. The seller analysis reports both the reviewed-order denominator used by the AI and all delivered orders with known dates.

In [1]:
import sys
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "olist_audit.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from olist_audit import audit_olist

audit = audit_olist()
headline = audit["headline_reproduction"]
checks = audit["checks"]
effect = checks["late_vs_on_time_score"]
denominators = checks["seller_denominator_comparison"]
before_after = pd.DataFrame([
    {"Measure": "Reviewed delivered orders", "AI analysis": headline["reviewed_delivered_orders"], "Corrected analysis": checks["review_selection"]["reviewed_orders"]},
    {"Measure": "Late rate", "AI analysis": headline["platform_late_rate_reviewed"], "Corrected analysis": checks["review_selection"]["all_delivered_late_rate"]},
    {"Measure": "Late minus on-time mean score", "AI analysis": effect["late_minus_on_time_mean"], "Corrected analysis": checks["adjusted_score_model"]["late_coefficient"]},
    {"Measure": "Unadjusted seller Wilson flags", "AI analysis": checks["seller_flags"]["wilson_flags_unadjusted"], "Corrected analysis": checks["seller_flags"]["bonferroni_survivors"]},
    {"Measure": "Eligible sellers (all delivered denominator)", "AI analysis": denominators["reviewed_order_seller_rates"]["eligible_seller_count"], "Corrected analysis": denominators["all_delivered_order_seller_rates"]["eligible_seller_count"]},
])
before_after

,Measure,AI analysis,Corrected analysis
0,Reviewed delivered orders,95824.000000,95824.000000
1,Late rate,0.079949,0.081124
2,Late minus on-time mean score,-1.729044,-1.712872
3,Unadjusted seller Wilson flags,75.000000,13.000000
4,Eligible sellers (all delivered denominator),623.000000,627.000000


## Estimates with uncertainty

The crude mean difference is reported with customer-clustered uncertainty. The adjusted model includes log total freight, item count, and customer state; it is not a causal estimate. Seller flags are assessed after correcting for the number of sellers tested.

In [2]:
print("Crude late-minus-on-time mean difference:", effect["late_minus_on_time_mean"])
print("Customer-clustered 95% CI:", effect["clustered_ci_95"])
print("Adjusted association:", checks["adjusted_score_model"])
print("Seller multiplicity results:", checks["seller_flags"])
print("Score means by lateness bucket:")
pd.DataFrame(headline["by_lateness_bucket"])

Crude late-minus-on-time mean difference: -1.729044476766842
Customer-clustered 95% CI: [-1.7669441053902528, -1.6911448481416553]
Adjusted association: {'late_coefficient': -1.7128716422291868, 'ci_95': [-1.7508735947662477, -1.6748696896921258], 'n_orders': 95824, 'covariates': ['log total freight', 'item count', 'customer state fixed effects'], 'limitations': ['observational association, not a causal effect', 'product category and geographic distance are unavailable', 'the model is restricted to reviewed orders']}
Seller multiplicity results: {'eligible_sellers_tested': 623, 'wilson_flags_unadjusted': 75, 'bh_survivors': 26, 'bonferroni_survivors': 13, 'p_value_reference_rate': 0.07994865586909333, 'seller_late_rate_ci_95': [0.09713132152543555, 0.14829191373747]}
Score means by lateness bucket:


,lateness_bucket,order_count,mean_review_score
0,on time / early,88163,4.294114
1,1-3 days late,2636,3.765933
2,4-7 days late,1773,2.316413
3,>7 days late,3252,1.727245


## Comparison with the committed reference

There is no blind `reference_analysis.ipynb` in this case folder. The numbers below compare the AI reproduction with the independent DuckDB reference output in `results.json`. The reference averages reviews per order and groups lateness by calendar dates; the AI keeps the latest review and uses elapsed timestamp days.

In [3]:
import json

case_results = json.loads((ROOT / "cases" / "03_olist_delivery_reviews" / "results.json").read_text(encoding="utf-8"))
reference = case_results["corrected_results"]["reference_comparison"]
reference_status = reference["reference_by_status"]
ai_status = reference["ai_by_status"]
pd.DataFrame([
    {"Measure": "Reviewed-order count", "AI reproduction": reference["ai_order_count"], "DuckDB reference": reference["reference_order_count"]},
    {"Measure": "Late mean score", "AI reproduction": ai_status["Late"]["mean_review_score"], "DuckDB reference": reference_status["Late"]["mean_review_score"]},
    {"Measure": "On-time mean score", "AI reproduction": ai_status["On time"]["mean_review_score"], "DuckDB reference": reference_status["On time"]["mean_review_score"]},
])

,Measure,AI reproduction,DuckDB reference
0,Reviewed-order count,95824.000000,95830.000000
1,Late mean score,2.565070,2.566562
2,On-time mean score,4.294114,4.294151


## Limitations

- Scores are unavailable for orders without a review. Those orders have a different observed late rate, so the score association applies to reviewed orders only.
- The files do not include product categories or geolocation needed for category and distance adjustment. Customer state and freight are only partial proxies.
- Delivery and review data are observational. Adjustment does not establish a causal effect.
- Orders with multiple sellers contribute to each seller's rate; seller-level rows overlap.
- Seller-level correlation is ecological: it describes seller averages and does not estimate the order-level relationship.

## Takeaway

Late orders in the reviewed sample received lower scores, and the difference remains after limited adjustment.
Some sellers have elevated late rates, but the count depends on correcting for many comparisons.
These results describe this dataset; they do not show that delay alone caused a lower score.